# NHS A&E Performance Analysis
**Author:** Husnain Zahoor  
**Date:** Aug 05, 2026  
**Dataset:** NHS A&E Quality Indicators (Provisional, December 2023 – December 2025)  
**Source URL:** https://tinyurl.com/NHS-Source-Data

---

### Context & Pipeline Position
This notebook (1 of 4) handles data loading, wrangling, information governance filtering (suppression), and consistency checks across three raw NHS source files. It executes a three-way join to produce the master cleaned dataset (`df_final.pkl`) for downstream analysis.

## Environment Setup
Mount Google Drive and define standard file paths. Run these three cells at the start of every session.

In [ ]:
# Mount Google Drive — run this first in every session
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# Define standard paths — reference these throughout the module
DATA_PATH   = '/content/drive/My Drive/Lumen/python-data-analytics/Data/'
OUTPUT_PATH = '/content/drive/My Drive/Lumen/python-data-analytics/Output/'

In [ ]:
# Verify setup — confirm all three CSV files are accessible
import os
print(os.listdir(DATA_PATH))

['aeqi_metadata.csv', 'aeqi_open_data_2025_12.csv', 'nhs_trust_reference.csv']


## Library Imports

In [ ]:
# Setup and Dependencies
import pandas as pd
import numpy as np

# FORCE PANDAS TO SHOW ALL COLUMNS HORIZONTALLY (NO WRAPPING)
pd.set_option('display.max_columns', None)  # Show every column
pd.set_option('display.width', 1000)        # Give the text engine plenty of horizontal wi

##Load dataset and Confirm


In [ ]:
df_aeqi = pd.read_csv(DATA_PATH + 'aeqi_open_data_2025_12.csv')

# Print shape and first 5 rows
print(df_aeqi.shape)
df_aeqi.head()

(94720, 7)


,ATTENDANCE_MONTH,ORG_CODE,ORG_NAME,MEASURE_ID,MEASURE_NAME,MEASURE_VALUE,SUPPRESSION
0,2024-09,8J094,BADGER LTD,AEQI053,TOTAL_TIME_MEDIAN_ADMITTED,268.0,NaN
1,2024-11,8J094,BADGER LTD,AEQI053,TOTAL_TIME_MEDIAN_ADMITTED,124.5,NaN
2,2025-02,8J094,BADGER LTD,AEQI053,TOTAL_TIME_MEDIAN_ADMITTED,31.0,NaN
3,2025-12,8J094,BADGER LTD,AEQI053,TOTAL_TIME_MEDIAN_ADMITTED,41.0,NaN
4,2023-12,AD903,URGENT CARE CENTRE,AEQI053,TOTAL_TIME_MEDIAN_ADMITTED,20.0,NaN


Loaded 94,720 rows × 7 columns. Column names match the NHS England
source structure: ATTENDANCE_MONTH, ORG_CODE, ORG_NAME, MEASURE_ID,
MEASURE_NAME, MEASURE_VALUE, SUPPRESSION. Row count confirms no rows
were lost or duplicated during load.

##Structural Inspection

In [ ]:
# Run each method below in its own line
# head(), tail(), shape, dtypes, info()
df_aeqi.head()

,ATTENDANCE_MONTH,ORG_CODE,ORG_NAME,MEASURE_ID,MEASURE_NAME,MEASURE_VALUE,SUPPRESSION
0,2024-09,8J094,BADGER LTD,AEQI053,TOTAL_TIME_MEDIAN_ADMITTED,268.0,NaN
1,2024-11,8J094,BADGER LTD,AEQI053,TOTAL_TIME_MEDIAN_ADMITTED,124.5,NaN
2,2025-02,8J094,BADGER LTD,AEQI053,TOTAL_TIME_MEDIAN_ADMITTED,31.0,NaN
3,2025-12,8J094,BADGER LTD,AEQI053,TOTAL_TIME_MEDIAN_ADMITTED,41.0,NaN
4,2023-12,AD903,URGENT CARE CENTRE,AEQI053,TOTAL_TIME_MEDIAN_ADMITTED,20.0,NaN


In [ ]:
df_aeqi.tail()

,ATTENDANCE_MONTH,ORG_CODE,ORG_NAME,MEASURE_ID,MEASURE_NAME,MEASURE_VALUE,SUPPRESSION
94715,2024-07,RP5,DONCASTER AND BASSETLAW TEACHING HOSPITALS NHS...,AEQI013,MSITAE_COMPARISON_ECDS_RATE,101.029780,NaN
94716,2025-07,RMP,TAMESIDE AND GLOSSOP INTEGRATED CARE NHS FOUND...,AEQI013,MSITAE_COMPARISON_ECDS_RATE,100.000000,NaN
94717,2024-09,RTP,SURREY AND SUSSEX HEALTHCARE NHS TRUST,AEQI013,MSITAE_COMPARISON_ECDS_RATE,100.301293,NaN
94718,2024-02,RTX,UNIVERSITY HOSPITALS OF MORECAMBE BAY NHS FOUN...,AEQI013,MSITAE_COMPARISON_ECDS_RATE,89.310206,NaN
94719,2024-02,ARN,LOCAL CARE DIRECT,AEQI011,MSITAE_COMPARISON_ECDS,0.000000,NaN


In [ ]:
df_aeqi.dtypes

,0
ATTENDANCE_MONTH,object
ORG_CODE,object
ORG_NAME,object
MEASURE_ID,object
MEASURE_NAME,object
MEASURE_VALUE,float64
SUPPRESSION,object


In [ ]:
df_aeqi.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 94720 entries, 0 to 94719
Data columns (total 7 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   ATTENDANCE_MONTH  94720 non-null  object 
 1   ORG_CODE          94720 non-null  object 
 2   ORG_NAME          93640 non-null  object 
 3   MEASURE_ID        94720 non-null  object 
 4   MEASURE_NAME      94720 non-null  object 
 5   MEASURE_VALUE     94720 non-null  float64
 6   SUPPRESSION       1041 non-null   object 
dtypes: float64(1), object(6)
memory usage: 5.1+ MB


Initial data quality observations:

1. ATTENDANCE_MONTH is stored as an object (text), not a date.
   This blocks chronological sorting and time-series analysis
   until converted with pd.to_datetime().

2. ORG_NAME has 1,080 missing values. These will be filled by
   joining against the Trust reference file, which holds the
   full organisation names.

3. SUPPRESSION is an object type with only 1,041 non-null values.
   It functions as a flag, not a calculation field — 'Y' marks
   rows withheld for patient privacy (small numbers rule). These
   rows must be handled deliberately during aggregation, not
   dropped, to avoid undercounting small or vulnerable services.

4. MEASURE_VALUE mixes counts, percentages, and time-based rates
   in one column, since all 23 KPIs share it. Aggregating this
   column directly (e.g. .mean()) without filtering by MEASURE_ID
   first produces mathematically meaningless results.

##Statistical Summary

In [ ]:
# Run describe() on the MEASURE_VALUE column
import pandas as pd
# This tells pandas to show numbers with 2 decimal places and no scientific notation
pd.options.display.float_format = '{:.2f}'.format

# Now run describe again
df_aeqi['MEASURE_VALUE'].describe()

,MEASURE_VALUE
count,94720.00
mean,6905.87
std,79089.42
min,-220.00
25%,79.00
50%,400.00
75%,3436.25
max,2416292.00


**MEASURE_VALUE distribution check:**

The mean (6,905.87) is much higher than the median (400.00). This
means the data is heavily right-skewed. Half of all values are 400
or below, but a small number of very large values (attendance counts
in the millions) pull the mean up. The median is the better number
to use here for "typical" performance.

The minimum value is -220, found in AEQI041 (Median Time to
Treatment). A negative wait time is not possible in real life, so
this is a data recording issue, not normal variation. It is a known
timestamp problem, also seen at Bradford Teaching Hospitals. These
rows are kept, not deleted, until the cause is confirmed.

The standard deviation (79,089.42) is almost 11 times the mean.
This happens because MEASURE_VALUE holds different types of numbers
in one column — small percentages next to large attendance counts.
This means the whole column cannot be analysed together. Each KPI
must be filtered out and analysed on its own.

##KPI Frequency

In [ ]:
#Run value_counts() on MEASURE_NAME
df_aeqi['MEASURE_NAME'].value_counts()

,count
MEASURE_NAME,
MSITAE_COMPARISON_ECDS,4487
MSITAE_COMPARISON_MSITAE,4486
MSITAE_COMPARISON_ECDS_RATE,4486
LEFT_DEPARTMENT_TOTAL,4404
LEFT_DEPARTMENT_ECDS_RATE,4403
TOTAL_TIME_95_NON_ADM,4302
TOTAL_TIME_DENOM_NON_ADM,4302
TOTAL_TIME_MEDIAN_NON_ADM,4302
TOTAL_TIME_95,4302


**Top 3 KPIs by row count:**
1. Total attendances in ECDS (AEQI011) — 4,487 rows
2. Total attendances in MSitAE (AEQI012) — 4,486 rows
3. ECDS attendances as % of MSitAE attendances (AEQI013) — 4,486 rows

These three lead because attendance volume is the most universally
reported measure — nearly every organisation submits it every month.
The theoretical maximum is 198 organisations × 25 months = 4,950 rows
per KPI. The actual count (~4,487) falls short of this by about 463
rows, meaning some organisations did not submit data for certain
months. This is a coverage gap, not a data error — it will be checked
against specific organisations and months in the consistency review.

##Suppression Analysis


In [ ]:
# Check suppression counts — use dropna=False
df_aeqi['SUPPRESSION'].value_counts(dropna=False)

,count
SUPPRESSION,
NaN,93679
Y,1041


In [ ]:
# TODO: Calculate suppression rate as a percentage of total rows
(df_aeqi['SUPPRESSION'].value_counts(dropna=False) / len(df_aeqi)) * 100

,count
SUPPRESSION,
NaN,98.90
Y,1.10


Suppressed rows represent data NHS England has withheld for patient
privacy. Where patient numbers are small enough to risk identification
— typically five or fewer cases — MEASURE_VALUE is set to 0 and
SUPPRESSION is flagged 'Y'. This is statistical disclosure control,
not a data error.

These rows are not dropped. They represent real activity at small
clinics and specialist departments, and must stay in the master
dataset. They are excluded from numerical calculations by filtering
SUPPRESSION != 'Y' before any aggregation.

The risk of skipping this filter: suppressed rows carry MEASURE_VALUE
= 0 as a placeholder, not a real result. Including them in a mean wait
time calculation artificially deflates the result — a Trust with
several suppressed months would appear to have near-zero wait times,
masking genuine performance problems. The Head of A&E Operations would
receive a misleading picture of which Trusts need intervention.

##Rename all columns

In [ ]:
# Confirm original column names before renaming
df_aeqi.columns

Index(['ATTENDANCE_MONTH', 'ORG_CODE', 'ORG_NAME', 'MEASURE_ID', 'MEASURE_NAME', 'MEASURE_VALUE', 'SUPPRESSION'], dtype='object')

In [ ]:
# Notice the space inside the first quotes ' '
df_aeqi.columns = df_aeqi.columns.str.strip().str.upper().str.replace(' ', '_')
print(df_aeqi.columns.tolist())

['ATTENDANCE_MONTH', 'ORG_CODE', 'ORG_NAME', 'MEASURE_ID', 'MEASURE_NAME', 'MEASURE_VALUE', 'SUPPRESSION']


##Convert the date column

In [ ]:
df_aeqi.dtypes

,0
ATTENDANCE_MONTH,object
ORG_CODE,object
ORG_NAME,object
MEASURE_ID,object
MEASURE_NAME,object
MEASURE_VALUE,float64
SUPPRESSION,object


In [ ]:
df_aeqi['ATTENDANCE_MONTH'] = pd.to_datetime(df_aeqi['ATTENDANCE_MONTH'], format = '%Y-%m')
print(df_aeqi['ATTENDANCE_MONTH'].dtype)
print(df_aeqi['ATTENDANCE_MONTH'].min(), 'to', df_aeqi['ATTENDANCE_MONTH'].max())

datetime64[ns]
2023-12-01 00:00:00 to 2025-12-01 00:00:00


##Create a published-only subset

In [ ]:
df_aeqi.head()

,ATTENDANCE_MONTH,ORG_CODE,ORG_NAME,MEASURE_ID,MEASURE_NAME,MEASURE_VALUE,SUPPRESSION
0,2024-09-01,8J094,BADGER LTD,AEQI053,TOTAL_TIME_MEDIAN_ADMITTED,268.00,NaN
1,2024-11-01,8J094,BADGER LTD,AEQI053,TOTAL_TIME_MEDIAN_ADMITTED,124.50,NaN
2,2025-02-01,8J094,BADGER LTD,AEQI053,TOTAL_TIME_MEDIAN_ADMITTED,31.00,NaN
3,2025-12-01,8J094,BADGER LTD,AEQI053,TOTAL_TIME_MEDIAN_ADMITTED,41.00,NaN
4,2023-12-01,AD903,URGENT CARE CENTRE,AEQI053,TOTAL_TIME_MEDIAN_ADMITTED,20.00,NaN


In [ ]:
# 1. Filter to EXCLUDE 'Y' (Keep rows where suppression is NOT 'Y')
df_aeqi_published = df_aeqi[df_aeqi['SUPPRESSION'] != 'Y']

# 2. Print the shapes of both DataFrames
print("Original DataFrame shape:", df_aeqi.shape)
print("Filtered DataFrame shape:", df_aeqi_published.shape)

# 3. Calculate the percentage of rows removed
rows_removed = len(df_aeqi) - len(df_aeqi_published)
percentage_removed = (rows_removed / len(df_aeqi)) * 100
print(f"Rows removed: {rows_removed} ({percentage_removed:.2f}%)")


# Suppressed rows are NOT data errors, they are NHS statistical
# Disclosure control protecting small providers with low patient volumes.
# They are excluded from calculations but must be documented, not deleted.

Original DataFrame shape: (94720, 7)
Filtered DataFrame shape: (93679, 7)
Rows removed: 1041 (1.10%)


##Subset to the five most common KPIs

In [ ]:
df_aeqi['MEASURE_ID'].value_counts()

,count
MEASURE_ID,
AEQI011,4487
AEQI012,4486
AEQI013,4486
AEQI021,4404
AEQI022,4403
AEQI056,4302
AEQI059,4302
AEQI055,4302
AEQI052,4302


In [ ]:
# Removing the National Aggregation ENG:
df_aeqi = df_aeqi[df_aeqi['ORG_CODE'] !='ENG']
print(df_aeqi.shape)
print(df_aeqi['ORG_CODE'].nunique(),'Organizations Remain')

(94145, 7)
197 Organizations Remain


In [ ]:
top5_measures = df_aeqi['MEASURE_ID'].value_counts().nlargest(5).index.tolist()
df_aeqi_top5_measures = df_aeqi[df_aeqi['MEASURE_ID'].isin(top5_measures)]

print("Top 5 Measure Row Counts:", df_aeqi_top5_measures['MEASURE_ID'].value_counts())

Top 5 Measure Row Counts: MEASURE_ID
AEQI011    4462
AEQI013    4461
AEQI012    4461
AEQI021    4379
AEQI022    4378
Name: count, dtype: int64


##Check all columns for mixed data types

In [ ]:
# Check actual Python types within each column
for col in df_aeqi.columns:
    unique_types = df_aeqi[col].map(type).unique()
    print(f"{col}: {unique_types}")

ATTENDANCE_MONTH: [<class 'pandas._libs.tslibs.timestamps.Timestamp'>]
ORG_CODE: [<class 'str'>]
ORG_NAME: [<class 'str'> <class 'float'>]
MEASURE_ID: [<class 'str'>]
MEASURE_NAME: [<class 'str'>]
MEASURE_VALUE: [<class 'float'>]
SUPPRESSION: [<class 'float'> <class 'str'>]


We found 2 columns with diff data types first one is **SUPPRESSION** column and 2nd one is **ORG_NAME** column

In [ ]:
# Fill NaN in SUPPRESSION with empty string so that our column becomes consistently str
df_aeqi['SUPPRESSION'] = df_aeqi['SUPPRESSION'].fillna('')

# Verify the data type
print(df_aeqi['SUPPRESSION'].map(type).unique())

[<class 'str'>]


**Two columns** show mixed types. ORG_NAME contains str and float because
1,080 rows have no organisation name recorded — pandas represents
missing text as NaN, which is a float type. These nulls will be
recovered later via a join on ORG_CODE with df_trust. No action taken
here; the data is not lost.

SUPPRESSION contains str and float because blank cells (non-suppressed
rows) loaded as NaN. Fixed with fillna('') to make the column uniformly
string, so SUPPRESSION == 'Y' filters behave consistently across all
94,145 rows.

##Count and interpret missing values per column

In [ ]:
print(df_aeqi.isnull().sum())
print()
print(f"Total null values: {df_aeqi.isnull().sum().sum()}")
print(f"Total rows: {len(df_aeqi)}")
print(f"Null rate: {df_aeqi.isnull().sum().sum() / (len(df_aeqi) * len(df_aeqi.columns)):.4%}")

ATTENDANCE_MONTH       0
ORG_CODE               0
ORG_NAME            1080
MEASURE_ID             0
MEASURE_NAME           0
MEASURE_VALUE          0
SUPPRESSION            0
dtype: int64

Total null values: 1080
Total rows: 94145
Null rate: 0.1639%


1,080 genuine NaN values were found, all in the ORG_NAME column. All
other columns are complete.

Suppressed rows are not counted as missing values because they represent
a known, real-world event, not an absence of data. A 'Y' in the
SUPPRESSION column means a Trust did treat patients that month, but
numbers were too low to publish safely. NHS England withholds the
corresponding MEASURE_VALUE under statistical disclosure control rules,
to protect patient privacy and avoid unreliable estimates from small
denominators. The data was collected — it was deliberately not released.

A genuine NaN, like those in ORG_NAME, means the value was never
recorded or was lost in the reporting pipeline — absent, not withheld.

1,041 rows carry SUPPRESSION = 'Y', 1.11% of the dataset, concentrated
in low-volume Trusts where attendance figures fall below the
five-or-fewer disclosure threshold.

In [ ]:
# Count the suppressed rows:
suppressed = df_aeqi[df_aeqi['SUPPRESSION'] == 'Y']
print(f"Suppressed rows: {len(suppressed)}")
print(f"As % of total: {len(suppressed) / len(df_aeqi):.2%}")

# Which KPIs are most often suppressed?
print(suppressed['MEASURE_ID'].value_counts().head(10))

Suppressed rows: 1041
As % of total: 1.11%
MEASURE_ID
AEQI033    263
AEQI058    190
AEQI022    145
AEQI021    135
AEQI062    121
AEQI063     65
AEQI061     57
AEQI043     33
AEQI011     10
AEQI013     10
Name: count, dtype: int64


**AEQI033 (263 rows)** ambulance denominator. Heavily suppressed because many smaller A&Es receive very few ambulance arrivals per month. Low volume triggers the five-or-fewer rule.

**AEQI058** **(190 rows)** admitted patient denominator. Small hospitals admit very few patients directly from A&E, same reason.

**AEQI021/022 (135–145 rows)** patients leaving before being seen. In a quiet department, even 1–2 walkouts per month breaches the disclosure threshold.

**The pattern confirms:** suppression clusters in denominators and low-frequency events at small providers not random data loss. This is deliberate NHS disclosure control working as designed.

In [ ]:
# Check for negative MEASURE_VALUE
neg_vals = df_aeqi[df_aeqi['MEASURE_VALUE'] < 0]
print(f"Rows with negative MEASURE_VALUE: {len(neg_vals)}")
print(neg_vals[['ORG_NAME', 'MEASURE_ID', 'ATTENDANCE_MONTH', 'MEASURE_VALUE']].head(10).to_string(index=False))

Rows with negative MEASURE_VALUE: 62
                                             ORG_NAME MEASURE_ID ATTENDANCE_MONTH  MEASURE_VALUE
     BRADFORD TEACHING HOSPITALS NHS FOUNDATION TRUST    AEQI041       2024-03-01         -70.00
       SHERWOOD FOREST HOSPITALS NHS FOUNDATION TRUST    AEQI041       2025-10-01         -16.00
ASHFORD AND ST PETER'S HOSPITALS NHS FOUNDATION TRUST    AEQI031       2023-12-01          -3.00
ASHFORD AND ST PETER'S HOSPITALS NHS FOUNDATION TRUST    AEQI031       2024-01-01          -4.00
ASHFORD AND ST PETER'S HOSPITALS NHS FOUNDATION TRUST    AEQI031       2024-02-01          -5.00
     BRADFORD TEACHING HOSPITALS NHS FOUNDATION TRUST    AEQI031       2024-03-01        -220.00
ASHFORD AND ST PETER'S HOSPITALS NHS FOUNDATION TRUST    AEQI031       2024-03-01          -4.00
ASHFORD AND ST PETER'S HOSPITALS NHS FOUNDATION TRUST    AEQI031       2024-04-01          -2.00
               SURREY AND SUSSEX HEALTHCARE NHS TRUST    AEQI031       2024-04-01         

62 rows have negative MEASURE_VALUE, all in two time-based KPIs:
AEQI041 (Median Time to Treatment) and AEQI031 (Median Time to
Initial Assessment). A negative time is not clinically possible.

Root cause: a timestamp sequencing issue. The source system recorded
the treatment or assessment time before the arrival time, creating a
negative interval.

This is localised to specific Trusts — including Bradford Teaching
Hospitals, Sherwood Forest Hospitals, Ashford and St Peter's, and
Surrey and Sussex. This confirms a recording issue at those sites,
not a fault in this pipeline.

These rows are flagged, not deleted. The raw NHS data is preserved
as-is. Whether to exclude them from time-based calculations is a
decision for the Head of A&E Operations, not the analyst.

##Check for duplicate rows and interpret the result

In [ ]:
# Duplicates checking:
num_duplicates = df_aeqi.duplicated().sum()
print(f"Exact duplicate rows: {num_duplicates}")

Exact duplicate rows: 0


0 exact duplicate rows confirmed across 94,145 rows.

A duplicate here would mean the same Trust reported the same KPI
for the same month more than once — not a data entry error, but a
structural fault in NHS England's data pipeline.

The consequence is concrete: if Bradford Teaching Hospitals appeared
twice for AEQI051 (Median Total Time) in March 2024, that Trust's
waiting time would be double-counted in any regional aggregation —
distorting the North of England's performance picture. NHS England
applies de-duplication before publication, but confirming this
independently is a required audit step. You cannot assert data
quality you have not verified.

##Verify the composite unique key

In [ ]:
composite_key = ['ATTENDANCE_MONTH','ORG_CODE', 'MEASURE_ID' ]

composite_duplicte = df_aeqi.duplicated(subset=composite_key).sum()
print(f"Composite key duplictes: {composite_duplicte}")

Composite key duplictes: 0


In [ ]:
n_orgs = df_aeqi['ORG_CODE'].nunique()
n_kpis = df_aeqi['MEASURE_ID'].nunique()
n_months = df_aeqi['ATTENDANCE_MONTH'].nunique()

theoretical_max = n_orgs * n_kpis * n_months
actual_rows = len(df_aeqi)
coverage_pct = actual_rows / theoretical_max * 100

print(f"Unique orgs: {n_orgs}")
print(f"Unique KPIs: {n_kpis}")
print(f"Unique months: {n_months}")
print(f"Theoretical max rows: {theoretical_max}")
print(f"Actual rows: {actual_rows}")
print(f"Coverage: {coverage_pct:.1f}%")

Unique orgs: 197
Unique KPIs: 23
Unique months: 25
Theoretical max rows: 113275
Actual rows: 94145
Coverage: 83.1%


The composite key (ORG_CODE × ATTENDANCE_MONTH × MEASURE_ID) is unique
across all 94,145 rows — zero duplicates.

Coverage is 83.1% (94,145 rows against a theoretical maximum of 113,275
rows = 197 organisations × 23 KPIs × 25 months).

The 16.9% shortfall has two separate causes:

1. Not all KPIs apply to all organisations. Ambulance assessment KPIs
   (AEQI031–033) only appear where a department receives ambulance
   arrivals. This explains the low row counts seen in Lesson 4.

2. Some organisations did not submit data for certain months. These
   rows are simply missing, not suppressed.

This is different from suppression. Suppression (1,041 rows) means
data exists but is withheld for privacy. The shortfall means data was
never submitted. One is a privacy control, the other is a gap in
reporting — they need separate treatment in the audit log.

##Load all three files and validate join keys

In [ ]:
df_aeqi = pd.read_pickle(OUTPUT_PATH + 'df_aeqi_clean.pkl')
df_metadata = pd.read_csv(DATA_PATH + 'aeqi_metadata.csv')
df_trusts = pd.read_csv(DATA_PATH + 'nhs_trust_reference.csv')

print(f"shape of df_aeqi : ", df_aeqi.shape)
print(f"shape of df_metadata: ", df_metadata.shape)
print(f"Shape of trusts: ", df_trusts.shape)

shape of df_aeqi :  (94145, 7)
shape of df_metadata:  (23, 6)
Shape of trusts:  (273, 7)


In [ ]:
# Verify join keys
print("AEQI MEASURE_ID sample:  ", df_aeqi['MEASURE_ID'].unique()[:5])
print("Metadata MEASURE_ID sample:", df_metadata['MEASURE_ID'].unique()[:5])
print()
print("AEQI ORG_CODE sample:    ", df_aeqi['ORG_CODE'].unique()[:5])
print("Trusts ORG_CODE sample:   ", df_trusts['ORG_CODE'].unique()[:5])

AEQI MEASURE_ID sample:   ['AEQI053' 'AEQI054' 'AEQI058' 'AEQI055' 'AEQI056']
Metadata MEASURE_ID sample: ['AEQI011' 'AEQI012' 'AEQI013' 'AEQI021' 'AEQI022']

AEQI ORG_CODE sample:     ['8J094' 'AD903' 'AD913' 'AGQ' 'AH6']
Trusts ORG_CODE sample:    ['G6V2S' 'R0A' 'R0B' 'R0C' 'R0D']


Join 1: df_aeqi → df_metadata on MEASURE_ID
The performance file has 23 KPI codes with no descriptions. Joining
the metadata file adds plain-English definitions, data types, and
caveats — so every row is interpretable without a separate lookup file.

Join 2: df_aeqi → df_trusts on ORG_CODE
The performance file has organisation codes with no geography.
Joining the Trust reference adds NHS region, ICB code, and open/close
dates — needed for the regional comparisons in Pillars 1, 2, and 3.

Both joins use a left join to keep df_aeqi as the source of truth.
Row count before and after each join confirms no rows were added or
dropped: 94,145 rows in, 94,145 rows out.

##**Join 1** merge performance data with metadata

In [ ]:
df_merged = pd.merge(
    df_aeqi , df_metadata,
    on = 'MEASURE_ID' ,  how = 'left'
)

print(f"Before join row count:", df_aeqi.shape)
print(f"After join row count:", df_merged.shape)
print(f"Row counts are preserved:", df_aeqi.shape[0] == df_merged.shape[0])

Before join row count: (94145, 7)
After join row count: (94145, 12)
Row counts are preserved: True


In [ ]:
df_merged.head(3)

,ATTENDANCE_MONTH,ORG_CODE,ORG_NAME,MEASURE_ID,MEASURE_NAME_x,MEASURE_VALUE,SUPPRESSION,MEASURE_NAME_y,DESCRIPTION,DATA_TYPE,CAVEATS,SPECIFICATION
0,2024-09-01,8J094,BADGER LTD,AEQI053,TOTAL_TIME_MEDIAN_ADMITTED,268.00,,TOTAL_TIME_MEDIAN_ADMITTED,Median Total Time in A&E (in minutes) Admitted...,Decimal,Excludes attendances over 72 hours,Median difference (minutes) between ARRIVAL_TI...
1,2024-11-01,8J094,BADGER LTD,AEQI053,TOTAL_TIME_MEDIAN_ADMITTED,124.50,,TOTAL_TIME_MEDIAN_ADMITTED,Median Total Time in A&E (in minutes) Admitted...,Decimal,Excludes attendances over 72 hours,Median difference (minutes) between ARRIVAL_TI...
2,2025-02-01,8J094,BADGER LTD,AEQI053,TOTAL_TIME_MEDIAN_ADMITTED,31.00,,TOTAL_TIME_MEDIAN_ADMITTED,Median Total Time in A&E (in minutes) Admitted...,Decimal,Excludes attendances over 72 hours,Median difference (minutes) between ARRIVAL_TI...


Row count stayed the same after the left join with the metadata file. This is expected: a left join keeps every row from the main dataset, no matter what. Every KPI in the performance data had a matching entry in the metadata file, so no row was lost and no blank values were created.

Four new columns were added from the metadata file: DESCRIPTION, DATA_TYPE, CAVEATS, SPECIFICATION. A fifth column, MEASURE_NAME, existed in both files, so pandas automatically renamed them MEASURE_NAME_x and MEASURE_NAME_y to avoid a clash. The metadata version was removed, and the performance version was renamed back to MEASURE_NAME. Final column count: 11.

In [ ]:
df_merged = df_merged.rename(columns = {'MEASURE_NAME_x':'MEASURE_NAME'})
df_merged = df_merged.drop(columns = ['MEASURE_NAME_y'])
print()
print(f"Columns after tidy up:", df_merged.columns.tolist())
print("Shape:", df_merged.shape)


Columns after tidy up: ['ATTENDANCE_MONTH', 'ORG_CODE', 'ORG_NAME', 'MEASURE_ID', 'MEASURE_NAME', 'MEASURE_VALUE', 'SUPPRESSION', 'DESCRIPTION', 'DATA_TYPE', 'CAVEATS', 'SPECIFICATION']
Shape: (94145, 11)


In [ ]:
df_merged.head(3)

,ATTENDANCE_MONTH,ORG_CODE,ORG_NAME,MEASURE_ID,MEASURE_NAME,MEASURE_VALUE,SUPPRESSION,DESCRIPTION,DATA_TYPE,CAVEATS,SPECIFICATION
0,2024-09-01,8J094,BADGER LTD,AEQI053,TOTAL_TIME_MEDIAN_ADMITTED,268.00,,Median Total Time in A&E (in minutes) Admitted...,Decimal,Excludes attendances over 72 hours,Median difference (minutes) between ARRIVAL_TI...
1,2024-11-01,8J094,BADGER LTD,AEQI053,TOTAL_TIME_MEDIAN_ADMITTED,124.50,,Median Total Time in A&E (in minutes) Admitted...,Decimal,Excludes attendances over 72 hours,Median difference (minutes) between ARRIVAL_TI...
2,2025-02-01,8J094,BADGER LTD,AEQI053,TOTAL_TIME_MEDIAN_ADMITTED,31.00,,Median Total Time in A&E (in minutes) Admitted...,Decimal,Excludes attendances over 72 hours,Median difference (minutes) between ARRIVAL_TI...


##**Join 2** merge with Trust reference and diagnose unmatched rows

In [ ]:
df_full = pd.merge(
    df_merged, df_trusts , on = 'ORG_CODE', how = 'left'
)
print(f"before join row count:", df_merged.shape)
print(f"After join row count:", df_full.shape)
print(f"Row count preserve: ", df_merged.shape[0] == df_full.shape[0])

before join row count: (94145, 11)
After join row count: (94145, 17)
Row count preserve:  True


In [ ]:
df_full.head(3)

,ATTENDANCE_MONTH,ORG_CODE,ORG_NAME_x,MEASURE_ID,MEASURE_NAME,MEASURE_VALUE,SUPPRESSION,DESCRIPTION,DATA_TYPE,CAVEATS,SPECIFICATION,ORG_NAME_y,NHSER_CODE,REGION,ICB_CODE,OPEN_DATE,CLOSE_DATE
0,2024-09-01,8J094,BADGER LTD,AEQI053,TOTAL_TIME_MEDIAN_ADMITTED,268.00,,Median Total Time in A&E (in minutes) Admitted...,Decimal,Excludes attendances over 72 hours,Median difference (minutes) between ARRIVAL_TI...,NaN,NaN,NaN,NaN,NaN,NaN
1,2024-11-01,8J094,BADGER LTD,AEQI053,TOTAL_TIME_MEDIAN_ADMITTED,124.50,,Median Total Time in A&E (in minutes) Admitted...,Decimal,Excludes attendances over 72 hours,Median difference (minutes) between ARRIVAL_TI...,NaN,NaN,NaN,NaN,NaN,NaN
2,2025-02-01,8J094,BADGER LTD,AEQI053,TOTAL_TIME_MEDIAN_ADMITTED,31.00,,Median Total Time in A&E (in minutes) Admitted...,Decimal,Excludes attendances over 72 hours,Median difference (minutes) between ARRIVAL_TI...,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
df_full = df_full.rename(columns = {'ORG_NAME_x': 'ORG_NAME'})
df_full = df_full.drop(columns = ['ORG_NAME_y'])
print()
print(f"columns after tidy up : ", df_full.columns.tolist())
print("Shape: ", df_full.shape)


columns after tidy up :  ['ATTENDANCE_MONTH', 'ORG_CODE', 'ORG_NAME', 'MEASURE_ID', 'MEASURE_NAME', 'MEASURE_VALUE', 'SUPPRESSION', 'DESCRIPTION', 'DATA_TYPE', 'CAVEATS', 'SPECIFICATION', 'NHSER_CODE', 'REGION', 'ICB_CODE', 'OPEN_DATE', 'CLOSE_DATE']
Shape:  (94145, 16)


In [ ]:
df_full.head(3)

,ATTENDANCE_MONTH,ORG_CODE,ORG_NAME,MEASURE_ID,MEASURE_NAME,MEASURE_VALUE,SUPPRESSION,DESCRIPTION,DATA_TYPE,CAVEATS,SPECIFICATION,NHSER_CODE,REGION,ICB_CODE,OPEN_DATE,CLOSE_DATE
0,2024-09-01,8J094,BADGER LTD,AEQI053,TOTAL_TIME_MEDIAN_ADMITTED,268.00,,Median Total Time in A&E (in minutes) Admitted...,Decimal,Excludes attendances over 72 hours,Median difference (minutes) between ARRIVAL_TI...,NaN,NaN,NaN,NaN,NaN
1,2024-11-01,8J094,BADGER LTD,AEQI053,TOTAL_TIME_MEDIAN_ADMITTED,124.50,,Median Total Time in A&E (in minutes) Admitted...,Decimal,Excludes attendances over 72 hours,Median difference (minutes) between ARRIVAL_TI...,NaN,NaN,NaN,NaN,NaN
2,2025-02-01,8J094,BADGER LTD,AEQI053,TOTAL_TIME_MEDIAN_ADMITTED,31.00,,Median Total Time in A&E (in minutes) Admitted...,Decimal,Excludes attendances over 72 hours,Median difference (minutes) between ARRIVAL_TI...,NaN,NaN,NaN,NaN,NaN


After the merge, identify all rows where REGION is NaN. How many unique organisations are unmatched? Print their ORG_CODE and ORG_NAME.

In [ ]:
# Rows with no REGION match, these are non-Trust orgs
no_region = df_full[df_full['REGION'].isnull()]
print(f"Rows with no Trust match: {len(no_region)}")
print(f"Unique org codes with no match: {no_region['ORG_CODE'].nunique()}")
print()
print("Unmatched orgs:")
print(no_region[['ORG_CODE', 'ORG_NAME']].drop_duplicates().sort_values('ORG_CODE'))

Rows with no Trust match: 15486
Unique org codes with no match: 56

Unmatched orgs:
      ORG_CODE                                           ORG_NAME
0        8J094                                         BADGER LTD
12452      91Q                      NHS KENT AND MEDWAY ICB - 91Q
72009    ACH01                        WHITSTABLE MEDICAL PRACTICE
4        AD903                                 URGENT CARE CENTRE
9        AD913                               BECKENHAM BEACON UCC
13         AGQ                NEMS COMMUNITY BENEFIT SERVICES LTD
16         AH6                URGENT CARE CENTRE- HURLEY GROUP HQ
21         ANH                                     MALLING HEALTH
26       AQN04                                  PHL LYMINGTON UTC
2556       ARN                                  LOCAL CARE DIRECT
31         AXA                    TOWER HAMLETS GP CARE GROUP CIC
34         AXG                                                NaN
37663      AXG                            WILTSHIRE HEALTH

Around 56 organisations have no match in the Trust reference file.
These are providers reporting A&E data but not registered as NHS
Trusts — Urgent Treatment Centres, GP out-of-hours cooperatives,
and Integrated Care Boards reporting aggregate figures.

This is not a data quality problem. The Trust reference file only
covers registered NHS Trusts by design, so these providers falling
outside it is expected, not an error. Treating it as a business
logic decision — these organisations are simply a different
provider type — rather than a join failure.

##Filter to active NHS Trusts and define df_final

In [ ]:
# Filter to active NHS Trusts only
df_final = df_full[
    df_full['REGION'].notna() &            # remove non-Trust orgs
    df_full['CLOSE_DATE'].isna() &        # remove closed Trusts
    (df_full['ORG_CODE'] != 'ENG')       # confirm ENGLAND aggregate excluded
].copy()

print(f"df_full rows:  {len(df_full)}")
print(f"df_final rows: {len(df_final)}")
print(f"Rows excluded: {len(df_full) - len(df_final)}")
print()
print(f"Unique Trusts in df_final: {df_final['ORG_CODE'].nunique()}")
print(f"Unique regions: {df_final['REGION'].nunique()}")
print(f"Unique KPIs: {df_final['MEASURE_ID'].nunique()}")
print(f"Date range: {df_final['ATTENDANCE_MONTH'].min()} to {df_final['ATTENDANCE_MONTH'].max()}")

df_full rows:  94145
df_final rows: 78386
Rows excluded: 15759

Unique Trusts in df_final: 140
Unique regions: 7
Unique KPIs: 23
Date range: 2023-12-01 00:00:00 to 2025-12-01 00:00:00


**Final population of df_final:**

The final analytical dataset contains 78,386 rows, covering:

140 active NHS Trusts (Trusts that actively submitted A&E performance data)
7 NHS Regions
23 KPIs (A&E performance metrics)
25 months of data (December 2023 to December 2025)

Note on Trust count: The NHS Trust reference file lists 205 active Trusts, but only 140 appear in the final dataset. This is because the join starts from the performance file — Trusts that don't submit A&E data (for example, mental health or community Trusts) are naturally excluded. This is expected and flagged here for stakeholder awareness.

**Three exclusions were applied to reach this final population:**

ENGLAND aggregate (ENG) removed — this row is a national rollup, not an individual provider. Keeping it would double-count activity and distort Trust-level and regional averages.
Non-Trust providers removed — 56 organisations (UTCs, GP cooperatives, ICBs) are not registered NHS Trusts and fall outside the scope of Trust-level analysis.
Closed Trusts removed — 68 Trusts in the reference file have a recorded close date. The analysis covers active providers only, since closed Trusts would have incomplete submission histories.

In [ ]:
# Verify why we have 140 trusts instead of 205
active_trusts_in_register = df_trusts[df_trusts['CLOSE_DATE'].isna()]['ORG_CODE'].nunique()
active_trusts_in_data = df_final['ORG_CODE'].nunique()

print(f"Active Trusts in registry master file: {active_trusts_in_register}")
print(f"Active Trusts that actually run an A&E: {active_trusts_in_data}")
print(f"Specialized Trusts with no A&E data:    {active_trusts_in_register - active_trusts_in_data}")

Active Trusts in registry master file: 205
Active Trusts that actually run an A&E: 140
Specialized Trusts with no A&E data:    65


In [ ]:
# Population flow summary this becomes Sheet 2 of your Excel report
print("=== POPULATION FLOW ===")
print(f"Source file (all orgs, all months): 94,720 rows")
print(f"After removing ENGLAND aggregate:   {94720 - 575} rows")
print(f"After left join with Trust ref:     {len(df_full)} rows")
print(f"After filtering to active Trusts:   {len(df_final)} rows (df_final)")

=== POPULATION FLOW ===
Source file (all orgs, all months): 94,720 rows
After removing ENGLAND aggregate:   94145 rows
After left join with Trust ref:     94145 rows
After filtering to active Trusts:   78386 rows (df_final)


##Export df_final and verify the export

In [ ]:
# Primary export — pickle preserves all dtypes
df_final.to_pickle(OUTPUT_PATH + 'df_final.pkl')
print("Saved: df_final.pkl")
print(f"df_final Shape: {df_final.shape}")

# Backup export — CSV for human-readable reference
df_final.to_csv(OUTPUT_PATH + 'df_final_backup.csv', index=False)
print("Saved: df_final_backup.csv")

Saved: df_final.pkl
df_final Shape: (78386, 16)
Saved: df_final_backup.csv


#**END**